# Geometry inside attention — corrected rerun of the B3 study (T4×2, two shards)

The first B3 study modified FreeFine's Temporal Contextal Attention (TCA) masks, but three
problems meant it did not test the intended mechanisms:

1. **Head layout of the released masks.** The masks are built per branch
   `[uncon_edit, uncon_ref, con_edit, con_ref]` and copied to the 8 heads with `.repeat`,
   while queries are ordered branch-major. As a result the object mask reaches only heads
   0, 2, 4, 6 of *every* branch — half of the edit heads, and half of the reference heads.
2. **G1 / G2 reached the reference branch** because they edited the whole mask tensor
   (and G1 also warped the reference branch's own keys).
3. **R²-TCA warped the reference image but not the reference mask.**
4. **G2's bias was about one token cell off** for rotations and reductions, because token
   centres were not converted consistently; G1's warp was within 0.13 cells and is unchanged
   in substance.

This notebook fixes all four, behind flags that are off by default:

| Arm | What it is |
|---|---|
| `c0` | corrected layout only (every edit head masked, reference untouched) |
| `c0_g2_l4`, `c0_g2_l8` | + G2 correspondence bias, λ = 4 / 8, σ = 1.5 cells, edit branches only |
| `c0_g1` | + G1 reference keys/values warped to the target pose, edit branches only |
| `c0_g1_g3` | + G3 validity gate on the TCA weight (radius 2 cells) |
| `c0_r2` | + R²-TCA: reference image **and** reference mask at the target pose |

Every arm is scored on the same 200-edit development subset, against the stored released
baseline (and the benchmark's coarse composite in shard A), with the official metric code,
seeded (MD and KD reproducible).

**Before generating anything** the notebook (i) re-runs the CPU test suite against the patched
code in the generation environment, and (ii) regenerates 5 cases with every flag off and
checks they reproduce the stored baseline images.

### How to run
Run the notebook **twice, on two accounts, in parallel**: set `SHARD = "A"` in cell 1 on one
and `SHARD = "B"` on the other.

| Shard | Generates | Scores |
|---|---|---|
| A | `c0`, `c0_g2_l4`, `c0_g2_l8` | baseline, coarse, the three arms |
| B | `c0_g1`, `c0_g1_g3`, `c0_r2` | baseline, the three arms |

Every geometry arm is built on the corrected layout, so its reference point is `c0` (from
shard A); the released baseline is scored in both shards, and both shards must reproduce the
earlier values of the baseline exactly (checked at the end), which makes the two shards
directly comparable.

### Inputs to attach (4 datasets, as in the June notebooks)
`freefine-geobench2d-bggen`, `geobench2d-metrics-subset`, `geobench2d-coarse-img`,
`freefine-sample-metadata`.

### Settings
Accelerator **GPU T4 ×2**, Internet **ON**, then *Save Version → Save & Run All (Commit)*.
Expected: ~15 min environments + ~4 h generation + ~1 h metrics per shard.

### If a shard stops early
Attach that saved version's output as an extra input and run it again: finished arms and
finished metric jobs are picked up, not repeated.

### Output to download
`/kaggle/working/gafix_results_<SHARD>.json` (and the `.csv` next to it).


In [1]:
# ===== 1. Configuration, clone and pin the FreeFine source =====
import os, time, subprocess, shutil
NB_START = time.time()
SHARD = "A"            # <-- "A" on one account, "B" on the other
COMMIT = "4c9fdb971572b32edbeac13464659274c28decbb"
WORK = "/kaggle/working"

C0 = {"FF_MASK_LAYOUT": "branch"}
ARMS_ALL = {
    "A": [("c0",       dict(C0)),
          ("c0_g2_l4", dict(C0, FF_GA=1, FF_GA_MODE="bias", FF_GA_LAMBDA=4, FF_GA_SIGMA=1.5)),
          ("c0_g2_l8", dict(C0, FF_GA=1, FF_GA_MODE="bias", FF_GA_LAMBDA=8, FF_GA_SIGMA=1.5))],
    "B": [("c0_g1",    dict(C0, FF_GA=1, FF_GA_MODE="warp")),
          ("c0_g1_g3", dict(C0, FF_GA=1, FF_GA_MODE="warp", FF_GA_GATE=1, FF_GA_GATE_CELLS=2)),
          ("c0_r2",    dict(C0, FF_R2=1))],
}
assert SHARD in ARMS_ALL, SHARD
ARMS = ARMS_ALL[SHARD]
EVAL_REFS = ["baseline", "coarse"] if SHARD == "A" else ["baseline"]
OUT = f"{WORK}/gafix_{SHARD}"
os.makedirs(OUT, exist_ok=True)

subprocess.run(
    f"mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine FreeFine_pristine && "
    f"git clone -q https://github.com/CIawevy/FreeFine.git && cd FreeFine && git checkout -q {COMMIT}",
    shell=True, check=True)
os.makedirs("/kaggle/temp/FreeFine_pristine/evaluation", exist_ok=True)
shutil.copytree("/kaggle/temp/FreeFine/src", "/kaggle/temp/FreeFine_pristine/src")
shutil.copytree("/kaggle/temp/FreeFine/evaluation/FreeFine", "/kaggle/temp/FreeFine_pristine/evaluation/FreeFine")
print("✓ cloned + pinned FreeFine", COMMIT[:14], "| shard", SHARD, "| arms", [a for a, _ in ARMS])


✓ cloned + pinned FreeFine 4c9fdb971572b3 | shard A | arms ['c0', 'c0_g2_l4', 'c0_g2_l8']


In [2]:
%%bash
# ===== 2. Generation environment (as in every FreeFine run of this project) =====
set -e
pip install -q --root-user-action=ignore uv; uv python install 3.10.13
V=/kaggle/temp/freefine_env; PY=$V/bin/python
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" --index-url https://download.pytorch.org/whl/cu121
cd /kaggle/temp/FreeFine
uv pip install --python "$PY" -r requirements.txt || { grep -v '^xformers' requirements.txt>/tmp/r.txt; uv pip install --python "$PY" -r /tmp/r.txt; uv pip install --python "$PY" xformers; }
uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0 "setuptools<70"
"$PY" -c "import torch,diffusers,xformers,cv2; print('freefine_env OK',torch.__version__)"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 73.7 MB/s eta 0:00:00
freefine_env OK 2.1.1+cu121


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.62s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/freefine_env
Activate with: source /kaggle/temp/freefine_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/freefine_env
Resolved 18 packages in 623ms
 Downloaded networkx
 Downloaded torchvision
 Downloaded pillow
 Downloaded sympy
 Downloaded numpy
 Downloaded triton
 Downloaded torch
Prepared 18 packages in 26.13s
Installed 18 packages in 362ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.3.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.1+cu121
 + torchvision==0.16.1+cu121
 + triton==2.1.0
 + typing-extensions==4.16.0
 + urllib3==1.26.13
Using Python 3.10.13 environment at: /kaggle/temp/freefine_en

In [3]:
%%bash
# ===== 2. Metric environment (identical to the final metric runs) =====
set -e
pip install -q --root-user-action=ignore uv
uv python install 3.10.13
V=/kaggle/temp/metric_env
PY=$V/bin/python
REPO=/kaggle/temp/FreeFine
rm -rf "$V"
uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "$PY" "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/m.txt
uv pip install --python "$PY" -r /tmp/m.txt
uv pip install --python "$PY" "setuptools<70"
uv pip install --python "$PY" --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python "$PY" "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $V -path '*/site-packages/clip' -o -path '*open_clip' -type d); do
  cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null || true
done
echo "✓ metric_env OK"


✓ metric_env OK


Python 3.10.13 is already installed
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 407ms
 Downloaded torchaudio
 Downloaded torchvision
 Downloaded sympy
 Downloaded networkx
 Downloaded pillow
 Downloaded nvidia-nvjitlink-cu12
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded numpy
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded nvidia-curand-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded nvidia-cusparselt-cu12
 Downloaded triton
 Downloaded nvidia-cusolver-cu12
 Downloaded nvidia-cublas-cu12
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded torch
Prepared 27 packages in 29.15s
Installed 27 packages in 255ms
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.4

In [4]:
# ===== 4. Official metric code: the four patches used by every evaluation in this project =====
#   (a) args.3d -> getattr(args,'3d')         the released entry point does not compile otherwise
#   (b) global seed at the entry point        makes kernel distance reproducible (FF_METRIC_SEED)
#   (c) SD-2.1 weights from the sd2-community mirror (the original repository was withdrawn)
#   (d) keypoint metric reseeded at its start  identical to the seeding of all June evaluations
import os, pathlib, py_compile
MR = pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")

def patch(path, old, new, marker):
    s = path.read_text(encoding="utf-8")
    if marker in s:
        return
    assert s.count(old) == 1, f"anchor not found exactly once in {path.name}: {old!r}"
    path.write_text(s.replace(old, new, 1), encoding="utf-8")

patch(MR / "main.py", "args.3d", "getattr(args,'3d')", "getattr(args,'3d')")
patch(MR / "main.py",
      "    args = parser.parse_args()\n",
      "    args = parser.parse_args()\n"
      "    import random as _random, numpy as _np\n"
      "    try:\n"
      "        import torch as _torch\n"
      "    except Exception:\n"
      "        _torch=None\n"
      "    _metric_seed=int(os.environ.get('FF_METRIC_SEED','42'))\n"
      "    _random.seed(_metric_seed); _np.random.seed(_metric_seed)\n"
      "    if _torch is not None:\n"
      "        _torch.manual_seed(_metric_seed)\n"
      "        if _torch.cuda.is_available(): _torch.cuda.manual_seed_all(_metric_seed)\n",
      "FF_METRIC_SEED")
for f in [MR / "MD" / "mean_distance.py", MR / "MD" / "dift_sd.py"]:
    f.write_text(f.read_text(encoding="utf-8").replace("stabilityai/stable-diffusion-2-1",
                                                       "sd2-community/stable-diffusion-2-1"), encoding="utf-8")
patch(MR / "MD" / "mean_distance.py", "all_dist = []",
      "all_dist = []\n    import torch as _st, os as _os; "
      "_seed=int(_os.environ.get('FF_MD_SEED','42')); "
      "_st.manual_seed(_seed); _st.cuda.manual_seed_all(_seed)", "FF_MD_SEED")
for f in [MR / "main.py", MR / "MD" / "mean_distance.py", MR / "MD" / "dift_sd.py"]:
    py_compile.compile(str(f), doraise=True)
assert "stabilityai/stable-diffusion-2-1" not in (MR / "MD" / "dift_sd.py").read_text(encoding="utf-8")
print("✓ metric patches installed: entry point, global seed, SD-2.1 mirror, keypoint-metric seed")


✓ metric patches installed: entry point, global seed, SD-2.1 mirror, keypoint-metric seed


In [5]:
# ===== 4. PyTorch 2.6 / OpenAI CLIP compatibility (identical to the final metric run) =====
import glob, pathlib
_clip_py = glob.glob("/kaggle/temp/metric_env/lib/python3.10/site-packages/clip/clip.py")
assert len(_clip_py) == 1, _clip_py
_cp = pathlib.Path(_clip_py[0]); _cs = _cp.read_text(encoding="utf-8")
_old = 'state_dict = torch.load(opened_file, map_location="cpu")'
_new = 'state_dict = torch.load(opened_file, map_location="cpu", weights_only=False)'
if _old in _cs:
    _cp.write_text(_cs.replace(_old, _new, 1), encoding="utf-8")
elif _new not in _cs:
    raise AssertionError("Unexpected OpenAI CLIP loader source; refusing broad torch.load patch")
print("✓ OpenAI CLIP PyTorch-2.6 compatibility patch installed")


✓ OpenAI CLIP PyTorch-2.6 compatibility patch installed


In [6]:
%%bash
mkdir -p /kaggle/temp/gafix_src


In [7]:
%%writefile /kaggle/temp/gafix_src/ga_fix.py
"""ga_fix.py -- corrected TCA mask layout and edit-branch-only geometry mechanisms for FreeFine.

Every mechanism is controlled by an environment flag and is OFF by default. With all flags off,
each function returns its input unchanged or evaluates the released expression verbatim, so the
patched pipeline reproduces the released computation bit for bit.

  FF_MASK_LAYOUT = tiled | branch  head layout of the TCA masks ('tiled' is the released code)
  FF_GA          = 1               apply a geometry mechanism to the EDIT branches only
  FF_GA_MODE     = bias | warp     G2 correspondence bias | G1 warped reference keys/values
  FF_GA_LAMBDA   (4.0)             G2 strength
  FF_GA_SIGMA    (1.5)             G2 width, in token cells
  FF_GA_GATE     = 1               G3 validity gate on the TCA weight (edit branches)
  FF_GA_GATE_CELLS (2.0)           G3 radius, in token cells
  FF_R2          = 1               R2-TCA: reference image AND reference mask at the target pose

Conventions
  * Inside TCA the batch is [uncon_edit, uncon_ref, con_edit, con_ref] and queries are
    batch-major after head_to_batch_dim: query row k belongs to branch k // heads. The edit
    branches are therefore the rows [0, H) and [2H, 3H).
  * The affine map M (2x3) is the one re_edit_2d() builds, in cv2 pixel coordinates of the
    512 x 512 working images (pixel centres at integer coordinates). A token of an R x R
    attention map covers a (512/R)-pixel block; its centre is at (i + 0.5) * s - 0.5.
"""
import os

import numpy as np
import torch


# ----------------------------------------------------------------------------- configuration
def _env(name, default):
    return os.environ.get(name, default)


def layout():
    v = _env("FF_MASK_LAYOUT", "tiled")
    assert v in ("tiled", "branch"), v
    return v


def ga_on():
    return _env("FF_GA", "0") == "1"


def ga_mode():
    v = _env("FF_GA_MODE", "bias")
    assert v in ("bias", "warp"), v
    return v


def r2_on():
    return _env("FF_R2", "0") == "1"


# ----------------------------------------------------------------------------- mask layout
def layout_rows(t, heads):
    """(4, ...) per-branch tensor -> (4*heads, ...).

    'tiled'  : t.repeat(heads, 1, ...), exactly as released (row k gets branch k % 4)
    'branch' : t.repeat_interleave(heads, 0) (row k gets branch k // heads, matching the queries)
    """
    if layout() == "branch":
        return t.repeat_interleave(heads, dim=0)
    return t.repeat(*((heads,) + (1,) * (t.dim() - 1)))


def post_process(ctrl, mask):
    """Attention_Modulator.post_process_attn_mask with the layout switch."""
    mask.masked_fill_(mask == 0, torch.finfo(mask.dtype).min)
    mask.masked_fill_(mask == 1, 0)
    return layout_rows(mask, ctrl.heads)


def edit_slices(heads):
    return slice(0, heads), slice(2 * heads, 3 * heads)


# ----------------------------------------------------------------------------- geometry
def token_centres(R, size, device="cpu"):
    """(R*R, 2) cv2 pixel coordinates (x, y) of the token centres, row-major."""
    s = size / R
    c = (torch.arange(R, dtype=torch.float64, device=device) + 0.5) * s - 0.5
    yy, xx = torch.meshgrid(c, c, indexing="ij")
    return torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)


def apply_affine(M, P):
    M = torch.as_tensor(np.asarray(M, dtype=np.float64), device=P.device)
    return P @ M[:, :2].T + M[:, 2]


def inverse_affine(M):
    F = np.vstack([np.asarray(M, dtype=np.float64), [0.0, 0.0, 1.0]])
    return np.linalg.inv(F)[:2]


def _cache(ctrl):
    c = getattr(ctrl, "_gafix_cache", None)
    if c is None:
        c = ctrl._gafix_cache = {}
    return c


def _R(seq):
    R = int(round(seq ** 0.5))
    assert R * R == seq, f"non-square attention map: seq={seq}"
    return R


def sq_dist_cells(ctrl, seq):
    """(seq_target, seq_source) squared distance, in token cells, between each target token
    centre and the image under M of each source token centre."""
    key = ("d2", seq)
    c = _cache(ctrl)
    if key not in c:
        R, size = _R(seq), float(ctrl.geo_size)
        P = token_centres(R, size)
        mapped = apply_affine(ctrl.geo_M, P)
        c[key] = torch.cdist(P, mapped) ** 2 / (size / R) ** 2
    return c[key]


def g2_bias(ctrl, seq, device, dtype):
    """G2: B[p, q] = -||t_p - M s_q||^2 / (2 sigma^2), token cells (cached per case on device)."""
    sigma = float(_env("FF_GA_SIGMA", "1.5"))
    key = ("g2", seq, str(device), str(dtype), sigma)
    c = _cache(ctrl)
    if key not in c:
        c[key] = (-sq_dist_cells(ctrl, seq) / (2.0 * sigma * sigma)).to(device, dtype)
    return c[key]


def g1_grid(ctrl, seq, device):
    """grid_sample grid (1, R, R, 2) that pulls, for every target token, the source value at
    its pre-image under M (align_corners=False)."""
    key = ("grid", seq)
    c = _cache(ctrl)
    if key not in c:
        R, size = _R(seq), float(ctrl.geo_size)
        P = token_centres(R, size)
        src = apply_affine(inverse_affine(ctrl.geo_M), P)
        norm = 2.0 * (src + 0.5) / size - 1.0
        c[key] = norm.reshape(1, R, R, 2).to(torch.float32)
    return c[key].to(device)


def warp_field(ctrl, x, seq, mode="bilinear"):
    """Warp a (B, seq, d) token field from the source pose to the target pose."""
    R = _R(seq)
    B, _, d = x.shape
    grid = g1_grid(ctrl, seq, x.device).to(x.dtype).expand(B, R, R, 2)
    f = x.permute(0, 2, 1).reshape(B, d, R, R)
    o = torch.nn.functional.grid_sample(f, grid, mode=mode, padding_mode="zeros", align_corners=False)
    return o.reshape(B, d, seq).permute(0, 2, 1)


def ref_mask_tokens(ctrl, seq):
    m = ctrl.process_mask_before_attention(ctrl.fg_ref_mask, seq)[0]
    return m.reshape(-1).float()


def g1_column_mask(ctrl, seq, device, dtype):
    """(seq,) additive mask: 0 at target tokens whose pre-image is source object, -inf-like else."""
    key = ("colmask", seq, str(device), str(dtype))
    c = _cache(ctrl)
    if key not in c:
        m = ref_mask_tokens(ctrl, seq).to(device)
        w = warp_field(ctrl, m[None, :, None], seq, mode="nearest")[0, :, 0]
        cm = torch.zeros(seq, dtype=dtype, device=device)
        cm[w <= 0] = torch.finfo(dtype).min
        c[key] = cm
    return c[key]


def g3_validity(ctrl, seq, device, dtype):
    """(seq,) 1 where some source object token maps closer than FF_GA_GATE_CELLS to the target
    token (strict, as in the original June definition: best bias > -cells^2 / 2 sigma^2)."""
    cells = float(_env("FF_GA_GATE_CELLS", "2.0"))
    key = ("g3", seq, str(device), str(dtype), cells)
    c = _cache(ctrl)
    if key not in c:
        d2 = sq_dist_cells(ctrl, seq).clone()
        obj = ref_mask_tokens(ctrl, seq) > 0
        d2[:, ~obj.cpu()] = float("inf")
        c[key] = (d2.min(dim=1).values < cells * cells).to(device, dtype)
    return c[key]


# ----------------------------------------------------------------------------- TCA hooks
def fg_inputs(ctrl, cross_key, cross_value, fg_mask, seq):
    """Keys, values and mask for TCA's object attention, plus an optional per-token gate.

    With FF_GA off the inputs are returned untouched. Otherwise only the edit-branch rows are
    modified; the reference-branch rows keep their keys, values and mask exactly.
    """
    if not ga_on():
        return cross_key, cross_value, fg_mask, None
    H = ctrl.heads
    e0, e2 = edit_slices(H)
    fk, fv, fm = cross_key, cross_value, fg_mask          # fg_mask is rebuilt on every call
    if ga_mode() == "bias":
        B = g2_bias(ctrl, seq, fm.device, fm.dtype)
        lam = float(_env("FF_GA_LAMBDA", "4.0"))
        fm[e0].add_(B, alpha=lam)
        fm[e2].add_(B, alpha=lam)
    else:
        fk, fv = cross_key.clone(), cross_value.clone()
        for e in (e0, e2):
            fk[e] = warp_field(ctrl, cross_key[e], seq)
            fv[e] = warp_field(ctrl, cross_value[e], seq)
        cm = g1_column_mask(ctrl, seq, fm.device, fm.dtype)
        fm[e0] = cm[None, None, :]
        fm[e2] = cm[None, None, :]
    gate = g3_validity(ctrl, seq, fm.device, fm.dtype) if _env("FF_GA_GATE", "0") == "1" else None
    return fk, fv, fm, gate


def blend(ctrl, ref_hidden, self_hidden, gate):
    """TCA output. gate=None evaluates the released expression verbatim."""
    if gate is None:
        return ref_hidden * (ctrl.context_guidance) + self_hidden * (1 - ctrl.context_guidance)
    H = ctrl.heads
    cg = float(ctrl.context_guidance)
    g = torch.full((ref_hidden.shape[0], ref_hidden.shape[1], 1), cg,
                   dtype=ref_hidden.dtype, device=ref_hidden.device)
    for e in edit_slices(H):
        g[e, :, 0] = cg * gate
    return ref_hidden * g + self_hidden * (1 - g)


# ----------------------------------------------------------------------------- R2-TCA
def r2_ref(ctrl, ori_img):
    """Reference image for inversion: the source warped to the target pose when FF_R2=1."""
    if not r2_on() or getattr(ctrl, "geo_M", None) is None:
        return ori_img
    import cv2
    h, w = ori_img.shape[:2]
    return cv2.warpAffine(ori_img, np.asarray(ctrl.geo_M, dtype=np.float32), (w, h))


def r2_ref_mask(ctrl, fg_ref_mask, target_mask):
    """Reference object mask: the target mask (the source mask warped by M) when FF_R2=1."""
    if not r2_on():
        return fg_ref_mask
    return target_mask


Writing /kaggle/temp/gafix_src/ga_fix.py


In [8]:
%%writefile /kaggle/temp/gafix_src/apply_patches.py
import os, pathlib, py_compile


def _once(src, old, new, name):
    n = src.count(old)
    assert n == 1, f"{name}: anchor found {n} times"
    return src.replace(old, new, 1)


def _in_function(src, signature, old, new, name):
    """Replace `old` exactly once inside the body of the method whose line starts with `signature`."""
    a = src.index(signature)
    b = src.find("\n    def ", a + len(signature))
    b = len(src) if b < 0 else b
    body = src[a:b]
    return src[:a] + _once(body, old, new, name) + src[b:]


def apply(root, ga_fix_source):
    root = pathlib.Path(root)
    (root / "ga_fix.py").write_text(ga_fix_source, encoding="utf-8")

    # --- 1. benchmark script -> parametrised sweep script (June parametrisation, now asserted)
    P = root / "evaluation" / "FreeFine"
    s = (P / "freefine_batch_infer_2d.py").read_text(encoding="utf-8")
    s = _once(s, "sys.path.append('/data/Hszhu/FreeFine')", f"sys.path.append('{root.as_posix()}')", "sys.path")
    s = _once(s, 'pretrained_model_path = "/data/Hszhu/prompt-to-prompt/stable-diffusion-v1-5/"',
              'pretrained_model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"', "sd15 path")
    s = _once(s,
              '        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n'
              '        obj_label = ""\n'
              '        ori_mask = read_and_resize_mask(ori_mask_path)\n',
              '        ori_mask = read_and_resize_mask(ori_mask_path)\n'
              '        coarse_input, target_mask, _geoM = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n'
              '        obj_label = (case.get("obj_label","") if os.environ.get("FF_USE_PROMPT")=="1" else "")\n',
              "mask order + M")
    s = _once(s, '"guidance_scale": 7.5,', '"guidance_scale": float(os.environ.get("FF_GUIDANCE","7.5")),', "cfg")
    s = _once(s, '"start_step": 35,', '"start_step": int(os.environ.get("FF_START_STEP","35")),', "start step")
    s = _once(s, 'dataset_json = osp.join(dst_base, "annotations_2d.json")',
              'dataset_json = os.environ.get("FF_SUBSET_JSON", osp.join(dst_base,"annotations_2d.json"))', "subset")
    s = _once(s, 'dst_gen_dir = osp.join(dst_base, "Geo-Bench-2D/Gen_results_FreeFine_2d")',
              'dst_gen_dir = os.environ.get("FF_OUT_DIR", osp.join(dst_base,"Geo-Bench-2D/Gen_results_FreeFine_2d"))',
              "out dir")
    s = _once(s, 'base_dir = "/data/Hszhu/dataset/GeoBenchMeta/"', 'base_dir = "/kaggle/temp/GeoBenchMeta"', "base dir")
    s = _once(s, "    return final_image, transformed_mask.astype(np.uint8)*255",
              "    return final_image, transformed_mask.astype(np.uint8)*255, rotation_matrix", "return M")
    s = _once(s, "        generated_results = model.FreeFine_generation(**params)",
              "        model.controller.geo_M = _geoM; model.controller.geo_size = 512\n"
              "        model.controller._gafix_cache = {}\n"
              "        generated_results = model.FreeFine_generation(**params)", "per-case geometry")
    (P / "freefine_sweep_2d.py").write_text(s, encoding="utf-8")

    # --- 2. attention.py: mask layout (one builder) and TCA object attention (one method)
    A = root / "src" / "utils" / "attention.py"
    a = A.read_text(encoding="utf-8")
    sig = "    def prepare_various_attention_mask(self, b, seq, value):"
    a = _in_function(a, sig,
                     "        FG_mask = self.post_process_attn_mask(FG_mask)\n"
                     "        BG_mask = self.post_process_attn_mask(BG_mask)\n",
                     "        FG_mask = __import__('ga_fix').post_process(self, FG_mask)\n"
                     "        BG_mask = __import__('ga_fix').post_process(self, BG_mask)\n", "layout FG/BG")
    a = _in_function(a, sig,
                     "final_mask_fg = torch.cat((fg_retain_mask, mask_, fg_retain_mask, mask_)).repeat(self.heads, 1)",
                     "final_mask_fg = __import__('ga_fix').layout_rows(torch.cat((fg_retain_mask, mask_, fg_retain_mask, mask_)), self.heads)",
                     "layout target mask")
    a = _in_function(a, sig,
                     "final_mask_fg_st2 = torch.cat((fg_retain_mask_st2, mask_, fg_retain_mask_st2, mask_)).repeat(self.heads, 1)",
                     "final_mask_fg_st2 = __import__('ga_fix').layout_rows(torch.cat((fg_retain_mask_st2, mask_, fg_retain_mask_st2, mask_)), self.heads)",
                     "layout target mask st2")
    sig = "    def Temporal_contextal_attention(self, query, key, value, is_cross, place_in_unet):"
    a = _in_function(a, sig,
                     "        attn_out_fg = self.mask_attention(query, cross_key, cross_value, src_FG_mask)\n",
                     "        _gk, _gv, _gm, _gate = __import__('ga_fix').fg_inputs(self, cross_key, cross_value, src_FG_mask, sequence_length)\n"
                     "        attn_out_fg = self.mask_attention(query, _gk, _gv, _gm)\n", "object attention")
    a = _in_function(a, sig,
                     "            hidden_states = ref_hidden * (self.context_guidance) + self_hidden * (1 - self.context_guidance)\n",
                     "            hidden_states = __import__('ga_fix').blend(self, ref_hidden, self_hidden, _gate)\n", "blend")
    A.write_text(a, encoding="utf-8")

    # --- 3. model.py: R2-TCA reference image and reference mask
    Mp = root / "src" / "demo" / "model.py"
    m = Mp.read_text(encoding="utf-8")
    m = _once(m, "ref_img=ori_img,verbose=verbose)  # ndarray mask",
              "ref_img=__import__('ga_fix').r2_ref(self.controller, ori_img),verbose=verbose)  # ndarray mask", "r2 image")
    m = _once(m, "        self.controller.fg_ref_mask = fg_ref_mask.to(self.device)",
              "        self.controller.fg_ref_mask = __import__('ga_fix').r2_ref_mask(self.controller, fg_ref_mask, fg_retain_mask).to(self.device)",
              "r2 mask")
    Mp.write_text(m, encoding="utf-8")

    for f in [root / "ga_fix.py", P / "freefine_sweep_2d.py", A, Mp]:
        py_compile.compile(str(f), doraise=True)
    return True


Writing /kaggle/temp/gafix_src/apply_patches.py


In [9]:
%%writefile /kaggle/temp/gafix_src/test_gafix.py
"""CPU tests for ga_fix + apply_patches against the pinned FreeFine source (commit 4c9fdb97).

T0  every patch anchor is found exactly once and all patched files compile
T1  all flags off: patched TCA output is bit-identical to the released TCA output
T2  branch layout: every edit-branch head is masked, no reference-branch head is
T3  branch layout: reference-branch output equals plain self-attention (released: it does not)
T4  G2 bias: reference rows bit-identical to the corrected baseline; bias peaks at M(source)
T5  G1 warp: token-field warp agrees with cv2.warpAffine (the coarse-composite warp)
T6  G1 warp: reference rows bit-identical to the corrected baseline; edit rows change
T7  G3 gate: valid exactly where a source-object token maps within the radius
T8  R2: reference image = cv2 warp of the source; reference mask = target mask
T9  sweep script: re_edit_2d returns the affine map, identical to cv2's composition
"""
import ast, importlib.util, os, shutil, sys, tempfile, types
import numpy as np
import torch
import cv2

HERE = os.path.dirname(os.path.abspath(__file__))
# pristine FreeFine source; on Kaggle the notebook sets both variables and has already patched
CLONE = os.environ.get("GAFIX_PRISTINE",
                       r"C:\Users\grtza\AppData\Local\Temp\claude\C--Users-grtza\cb40de53-357f-429c-96c0-0e4b3b8397c7\scratchpad\FreeFine")
PATCHED = os.environ.get("GAFIX_PATCHED_ROOT")
sys.modules.setdefault("matplotlib", types.ModuleType("matplotlib"))
sys.modules.setdefault("matplotlib.pyplot", types.ModuleType("matplotlib.pyplot"))
FLAGS = ["FF_MASK_LAYOUT", "FF_GA", "FF_GA_MODE", "FF_GA_LAMBDA", "FF_GA_SIGMA", "FF_GA_GATE",
         "FF_GA_GATE_CELLS", "FF_R2"]
ok = lambda m: print("  PASS", m)


def set_flags(**kw):
    for f in FLAGS:
        os.environ.pop(f, None)
    for k, v in kw.items():
        os.environ[k] = str(v)


# ------------------------------------------------------------------ T0 apply the patches
print("T0 patches")
if PATCHED:
    ROOT = PATCHED
    ok("using the already-patched repository " + ROOT)
else:
    TMP = tempfile.mkdtemp(prefix="gafix_")
    ROOT = os.path.join(TMP, "FreeFine")
    shutil.copytree(os.path.join(CLONE, "src"), os.path.join(ROOT, "src"))
    shutil.copytree(os.path.join(CLONE, "evaluation", "FreeFine"), os.path.join(ROOT, "evaluation", "FreeFine"))
    sys.path.insert(0, HERE)
    import apply_patches
    apply_patches.apply(ROOT, open(os.path.join(HERE, "ga_fix.py"), encoding="utf-8").read())
    ok("all anchors found once; ga_fix.py, sweep script, attention.py, model.py compile")
sys.path.insert(0, ROOT)
import ga_fix


def load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


orig = load("attn_orig", os.path.join(CLONE, "src", "utils", "attention.py"))
patched = load("attn_patched", os.path.join(ROOT, "src", "utils", "attention.py"))

# ------------------------------------------------------------------ a realistic TCA call
H, R, D = 8, 32, 8
SEQ = R * R
torch.manual_seed(0)
Q = torch.randn(4, SEQ, H * D)
K = torch.randn(4, SEQ, H * D)
V = torch.randn(4, SEQ, H * D)
src_mask = torch.zeros(512, 512); src_mask[160:352, 64:224] = 1          # object on the left
M = cv2.getRotationMatrix2D((144.0, 256.0), -30.0, 1.0)                  # rotate about its centre...
M[0, 2] += 176.0                                                          # ...and move it right
tgt_np = cv2.warpAffine(src_mask.numpy().astype(np.uint8), M, (512, 512), flags=cv2.INTER_NEAREST)
tgt_mask = torch.from_numpy(tgt_np.astype(np.float32))


def controller(mod):
    c = mod.Attention_Modulator(start_layer=10)
    c.heads, c.scale, c.upcast_attention, c.upcast_softmax = H, D ** -0.5, False, False
    c.method, c.context_guidance = "tca", 0.7
    c.layer_idx = list(range(10, 16))
    c.cur_att_layer, c.num_att_layers = 20, 32          # layer 20 // 2 = 10 is modulated
    c.fg_ref_mask, c.fg_retain_mask, c.fg_retain_mask_st2 = src_mask, tgt_mask, tgt_mask
    c.geo_M, c.geo_size, c._gafix_cache = M, 512, {}
    return c


def tca(mod):
    c = controller(mod)
    with torch.no_grad():
        return c.Temporal_contextal_attention(Q.clone(), K.clone(), V.clone(), False, "up")


def branch_rows(x, b):          # (4, SEQ, H*D) -> rows of branch b
    return x[b]

# ------------------------------------------------------------------ T1 no-op
print("T1 all flags off reproduces the released computation")
set_flags()
out_rel, out_pat = tca(orig), tca(patched)
assert torch.equal(out_rel, out_pat), (out_rel - out_pat).abs().max()
ok(f"bit-identical output {tuple(out_pat.shape)}")

# ------------------------------------------------------------------ T2 layout
print("T2 head layout of the object mask")
obj = torch.nn.functional.interpolate(src_mask[None, None], size=(R, R), mode="nearest")[0, 0].flatten() > 0


def outside_mass(layout):
    set_flags(FF_MASK_LAYOUT=layout)
    c = controller(patched)
    fg, _, _, _, _ = c.prepare_various_attention_mask(4, SEQ, torch.zeros(1, SEQ, D))
    p = c.get_attention_scores(c.head_to_batch_dim(Q), c.head_to_batch_dim(K), fg)
    return torch.tensor([[float(p[b * H + h][:, ~obj].sum(1).mean()) for h in range(H)] for b in range(4)])


tiled, branch = outside_mass("tiled"), outside_mass("branch")
print("    released (tiled): edit heads masked =", (tiled[[0, 2]] < 1e-6).sum().item(), "/ 16;",
      "reference heads masked =", (tiled[[1, 3]] < 1e-6).sum().item(), "/ 16")
assert (branch[[0, 2]] < 1e-6).all() and (branch[[1, 3]] > 0.1).all()
ok("branch layout: 16/16 edit heads masked, 0/16 reference heads masked")

# ------------------------------------------------------------------ T3 reference untouched
print("T3 reference branch under the corrected layout")
def plain_self_attention(b):
    c = controller(patched)
    q, k, v = (c.head_to_batch_dim(x) for x in (Q, K, V))
    s = c.mask_attention(q[b * H:(b + 1) * H], k[b * H:(b + 1) * H], v[b * H:(b + 1) * H], None)
    return c.batch_to_head_dim(s)[0]

set_flags(FF_MASK_LAYOUT="branch"); out_c0 = tca(patched)
set_flags(FF_MASK_LAYOUT="tiled"); out_tiled = tca(patched)
for b in (1, 3):
    ref = plain_self_attention(b)
    assert torch.allclose(out_c0[b], ref, atol=1e-5), (out_c0[b] - ref).abs().max()
    d_rel = float((out_tiled[b] - ref).abs().max())
    print(f"    branch {b}: corrected max|diff| = {float((out_c0[b] - ref).abs().max()):.1e};"
          f" released max|diff| = {d_rel:.2f}")
ok("corrected layout leaves the reference branch equal to its own self-attention")

# ------------------------------------------------------------------ T4 G2
print("T4 G2 correspondence bias")
set_flags(FF_MASK_LAYOUT="branch", FF_GA=1, FF_GA_MODE="bias", FF_GA_LAMBDA=4, FF_GA_SIGMA=1.5)
out_g2 = tca(patched)
for b in (1, 3):
    assert torch.equal(out_g2[b], out_c0[b])
assert not torch.allclose(out_g2[0], out_c0[0])
c = controller(patched)
Bm = ga_fix.g2_bias(c, SEQ, "cpu", torch.float32)
P = ga_fix.token_centres(R, 512.0)
mapped = cv2.transform(P.numpy()[None].astype(np.float64), M)[0]            # cv2's own mapping
for p in torch.randint(0, SEQ, (50,)).tolist():
    q = int(Bm[p].argmax())
    dq = np.linalg.norm(mapped - P[p].numpy(), axis=1)
    assert abs(dq[q] - dq.min()) < 1e-6
ok("reference rows bit-identical to the corrected baseline; bias peaks at the cv2-mapped source token")

# ------------------------------------------------------------------ T5 G1 geometry vs cv2
print("T5 G1 token warp agrees with cv2.warpAffine")
s = 512 // R
img = np.zeros((512, 512), np.float32)
field = torch.randn(SEQ)
for t in range(SEQ):
    i, j = divmod(t, R)
    img[i * s:(i + 1) * s, j * s:(j + 1) * s] = float(field[t])
for name, Mt in [("translation by 5 tokens", np.float64([[1, 0, 5 * s], [0, 1, -3 * s]])),
                 ("rotation by 90 deg", cv2.getRotationMatrix2D((255.5, 255.5), -90.0, 1.0))]:
    c = controller(patched); c.geo_M = Mt; c._gafix_cache = {}
    warped_img = cv2.warpAffine(img, Mt, (512, 512), flags=cv2.INTER_NEAREST)
    expect = torch.from_numpy(warped_img[s // 2::s, s // 2::s].copy()).reshape(-1)
    got = ga_fix.warp_field(c, field[None, :, None], SEQ, mode="nearest")[0, :, 0]
    assert torch.equal(got, expect), (got - expect).abs().max()
    print(f"    {name}: exact match on all {SEQ} tokens")
ok("G1 uses the same geometry as the coarse composite")

# ------------------------------------------------------------------ T6 G1 edit-only
print("T6 G1 touches only the edit branches")
set_flags(FF_MASK_LAYOUT="branch", FF_GA=1, FF_GA_MODE="warp")
out_g1 = tca(patched)
for b in (1, 3):
    assert torch.equal(out_g1[b], out_c0[b])
assert not torch.allclose(out_g1[0], out_c0[0])
ok("reference rows bit-identical to the corrected baseline; edit rows change")

# ------------------------------------------------------------------ T7 G3
print("T7 G3 validity gate")
set_flags(FF_MASK_LAYOUT="branch", FF_GA=1, FF_GA_MODE="warp", FF_GA_GATE=1, FF_GA_GATE_CELLS=2)
c = controller(patched)
v = ga_fix.g3_validity(c, SEQ, "cpu", torch.float32)
tgt_tok = torch.nn.functional.interpolate(tgt_mask[None, None], size=(R, R), mode="nearest")[0, 0].flatten() > 0
d = np.linalg.norm(P.numpy()[:, None, :] - mapped[None, obj.numpy(), :], axis=2).min(1) / s
assert torch.equal(v > 0, torch.from_numpy(d < 2.0))
print(f"    valid tokens: {int(v.sum())}; target-mask tokens covered: {int((v[tgt_tok] > 0).sum())}/{int(tgt_tok.sum())}")
out_g3 = tca(patched)
for b in (1, 3):
    assert torch.equal(out_g3[b], out_c0[b])
ok("gate equals the brute-force definition; reference rows untouched")

# ------------------------------------------------------------------ T8 R2
print("T8 R2-TCA")
c = controller(patched)
set_flags(FF_R2=1)
im = (np.random.default_rng(0).random((512, 512, 3)) * 255).astype(np.uint8)
assert np.array_equal(ga_fix.r2_ref(c, im), cv2.warpAffine(im, M.astype(np.float32), (512, 512)))
assert ga_fix.r2_ref_mask(c, "SRC", "TGT") == "TGT"
set_flags()
assert ga_fix.r2_ref(c, im) is im and ga_fix.r2_ref_mask(c, "SRC", "TGT") == "SRC"
ok("reference image warped by M and reference mask = target mask when on; untouched when off")

# ------------------------------------------------------------------ T9 script geometry
print("T9 sweep script returns the coarse-composite map")
src_code = open(os.path.join(ROOT, "evaluation", "FreeFine", "freefine_sweep_2d.py"), encoding="utf-8").read()
fn = next(n for n in ast.parse(src_code).body if isinstance(n, ast.FunctionDef) and n.name == "re_edit_2d")
ns = {"np": np, "cv2": cv2}
exec(compile(ast.Module(body=[fn], type_ignores=[]), "re_edit_2d", "exec"), ns)
img3 = (np.random.default_rng(1).random((512, 512, 3)) * 255).astype(np.uint8)
m2 = np.zeros((512, 512), np.uint8); m2[100:300, 150:260] = 255
final, tmask, Mret = ns["re_edit_2d"](img3, m2, [20, -15, 0, 0, 0, 25, 1.3, 1.3, 1], np.zeros_like(img3))
assert np.array_equal(tmask, cv2.warpAffine(m2, Mret, (512, 512), flags=cv2.INTER_NEAREST).astype(bool).astype(np.uint8) * 255)
ok("re_edit_2d returns the exact matrix used to build the composite")

print("\nALL TESTS PASSED")


Writing /kaggle/temp/gafix_src/test_gafix.py


In [10]:
# ===== 7. Apply the corrected mechanisms to the pinned source, then re-run the test suite =====
import os, sys, subprocess
sys.path.insert(0, "/kaggle/temp/gafix_src")
import apply_patches
apply_patches.apply("/kaggle/temp/FreeFine", open("/kaggle/temp/gafix_src/ga_fix.py", encoding="utf-8").read())
print("✓ patches applied (all anchors found exactly once, all files compile)")

env = os.environ.copy()
env.update({"GAFIX_PRISTINE": "/kaggle/temp/FreeFine_pristine", "GAFIX_PATCHED_ROOT": "/kaggle/temp/FreeFine",
            "CUDA_VISIBLE_DEVICES": "", "PYTHONIOENCODING": "utf-8"})
r = subprocess.run(["/kaggle/temp/freefine_env/bin/python", "/kaggle/temp/gafix_src/test_gafix.py"],
                   env=env, capture_output=True, text=True)
print(r.stdout[-6000:])
if r.returncode != 0 or "ALL TESTS PASSED" not in r.stdout:
    print(r.stderr[-4000:])
    raise RuntimeError("self-test failed in the generation environment: do not generate")


✓ patches applied (all anchors found exactly once, all files compile)
T0 patches
  PASS using the already-patched repository /kaggle/temp/FreeFine
T1 all flags off reproduces the released computation
  PASS bit-identical output (4, 1024, 64)
T2 head layout of the object mask
    released (tiled): edit heads masked = 8 / 16; reference heads masked = 8 / 16
  PASS branch layout: 16/16 edit heads masked, 0/16 reference heads masked
T3 reference branch under the corrected layout
    branch 1: corrected max|diff| = 3.0e-08; released max|diff| = 0.72
    branch 3: corrected max|diff| = 3.0e-08; released max|diff| = 0.43
  PASS corrected layout leaves the reference branch equal to its own self-attention
T4 G2 correspondence bias
  PASS reference rows bit-identical to the corrected baseline; bias peaks at the cv2-mapped source token
T5 G1 token warp agrees with cv2.warpAffine
    translation by 5 tokens: exact match on all 1024 tokens
    rotation by 90 deg: exact match on all 1024 tokens
  PA

In [11]:
# ===== 8. Data, development subset (identical seeded procedure), reference sets =====
import os, glob, json, csv, random, shutil
from collections import defaultdict, Counter
GEO = "/kaggle/temp/GeoBenchMeta"; os.makedirs(f"{GEO}/Geo-Bench-2D", exist_ok=True)
CACHE = next(c for c in glob.glob("/kaggle/input/**/Geo-Bench-2D", recursive=True) if os.path.isdir(f"{c}/source_img"))
COARSE = glob.glob("/kaggle/input/**/coarse_img/*/*/*.png", recursive=True)[0].split("/coarse_img/")[0] + "/coarse_img"
GENBASE = glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup", recursive=True)[0]
IB = os.path.dirname(os.path.dirname(os.path.dirname(
    glob.glob("/kaggle/input/**/inp_img_blended/**/inp_img.png", recursive=True)[0])))
ANNs = glob.glob("/kaggle/input/**/annotation_2d.json", recursive=True)[0]
META = glob.glob("/kaggle/input/**/sample_metadata.csv", recursive=True)[0]
for nm in ["source_img", "source_mask", "target_mask", "source_img_full_v2"]:
    d = f"{GEO}/Geo-Bench-2D/{nm}"; os.path.exists(d) or os.symlink(f"{CACHE}/{nm}", d)
for nm, sc in [("coarse_img", COARSE), ("inp_img_blended", IB)]:
    d = f"{GEO}/Geo-Bench-2D/{nm}"; os.path.exists(d) or os.symlink(sc, d)
shutil.copy(ANNs, f"{GEO}/annotation_2d.json"); ann = json.load(open(f"{GEO}/annotation_2d.json"))
shutil.copy(ANNs, f"{GEO}/annotations_2d.json")

meta = [r for r in csv.DictReader(open(META)) if os.path.exists(f"{IB}/{r['da_n']}/{r['ins_id']}/inp_img.png")]
random.seed(42); cells = defaultdict(list)
for r in meta: cells[(r["edit_type"], r["difficulty"])].append(r)
keys = sorted(cells); per = 200 // len(keys); picked = []
for k in keys:
    pool = cells[k][:]; random.shuffle(pool); picked += pool[:per]
ch = {(r["da_n"], r["ins_id"], r["case_id"]) for r in picked}
left = [r for r in meta if (r["da_n"], r["ins_id"], r["case_id"]) not in ch]; random.shuffle(left)
for r in left:
    if len(picked) >= 200: break
    picked.append(r)
picked = picked[:200]; json.dump(picked, open(f"{GEO}/subset_meta.json", "w"))
comp = Counter(r["edit_type"] for r in picked)
nrh = sum(r["edit_type"] == "rotate" and r["difficulty"] == "hard" for r in picked)
nsh = sum(r["edit_type"] == "resize" and r["difficulty"] == "hard" for r in picked)
assert comp == {"move": 67, "rotate": 66, "resize": 67} and (nrh, nsh) == (22, 23), (comp, nrh, nsh)
print("✓ development subset re-derived:", dict(comp), "| hard rotate", nrh, "| hard resize", nsh)

gsub = {}
for r in picked:
    d, i, e = r["da_n"], r["ins_id"], r["case_id"]; lf = dict(ann[d]["instances"][i][e])
    lf["ori_img_path"] = os.path.join(GEO, lf["ori_img_path"]); lf["ori_mask_path"] = os.path.join(GEO, lf["ori_mask_path"])
    gsub.setdefault(d, {"instances": {}})["instances"].setdefault(i, {})[e] = lf
json.dump(gsub, open(f"{GEO}/gen_subset.json", "w"))
os.makedirs(f"{GEO}/gen_eval", exist_ok=True)
for nm, sc in [("baseline", GENBASE), ("coarse", f"{GEO}/Geo-Bench-2D/coarse_img")]:
    d = f"{GEO}/gen_eval/{nm}"
    if os.path.islink(d): os.remove(d)
    os.symlink(sc, d)
print("✓ reference sets linked: baseline, coarse")


✓ development subset re-derived: {'move': 67, 'resize': 67, 'rotate': 66} | hard rotate 22 | hard resize 23
✓ reference sets linked: baseline, coarse


In [12]:
# ===== 9. Determinism gate: every flag off must reproduce the stored baseline images =====
import os, json, socket, subprocess, glob, numpy as np
from PIL import Image
P = "/kaggle/temp/FreeFine/evaluation/FreeFine"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("(no HF_TOKEN secret attached; public mirrors do not need it)")
gs = json.load(open(f"{GEO}/gen_subset.json")); val = {}; n = 0
for d, da in gs.items():
    for i, ins in da["instances"].items():
        for e in ins:
            val.setdefault(d, {"instances": {}})["instances"].setdefault(i, {})[e] = ins[e]; n += 1
            if n >= 5: break
        if n >= 5: break
    if n >= 5: break
json.dump(val, open(f"{GEO}/val5.json", "w"))
import shutil; shutil.rmtree("/kaggle/temp/val5", ignore_errors=True)   # never compare stale images
os.makedirs("/kaggle/temp/val5", exist_ok=True)

def gen_env(outdir, subset, **flags):
    env = os.environ.copy()
    for f in ["FF_MASK_LAYOUT", "FF_GA", "FF_GA_MODE", "FF_GA_LAMBDA", "FF_GA_SIGMA", "FF_GA_GATE",
              "FF_GA_GATE_CELLS", "FF_R2"]:
        env.pop(f, None)
    env.update({"PATH": "/kaggle/temp/freefine_env/bin:" + env["PATH"], "PYTHONUNBUFFERED": "1",
                "TOKENIZERS_PARALLELISM": "false", "HF_HOME": "/kaggle/temp/hf", "NCCL_P2P_DISABLE": "1",
                "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "FF_SUBSET_JSON": subset, "FF_OUT_DIR": outdir})
    env.update({k: str(v) for k, v in flags.items()})
    return env

s = socket.socket(); s.bind(("", 0)); port = s.getsockname()[1]; s.close()
r = subprocess.run(["/kaggle/temp/freefine_env/bin/torchrun", "--nproc_per_node=1", "--master-port", str(port),
                    "freefine_sweep_2d.py"], cwd=P, env=gen_env("/kaggle/temp/val5", f"{GEO}/val5.json"),
                   capture_output=True, text=True)
imgs = sorted(glob.glob("/kaggle/temp/val5/**/*.png", recursive=True))
if not imgs:
    print((r.stdout + r.stderr)[-3000:]); raise SystemExit("gate produced no images")
diffs = [float(np.abs(np.array(Image.open(f).convert("RGB"), float) - np.array(
    Image.open(f"{GENBASE}/{os.path.relpath(f, '/kaggle/temp/val5')}").convert("RGB").resize(Image.open(f).size), float)).mean())
         for f in imgs]
print("gate mean|Δ| per case:", [round(x, 3) for x in diffs])
assert max(diffs) < 1.0, "patched pipeline does not reproduce the baseline"
print("✓ gate passed" + ("" if max(diffs) == 0.0 else "  (WARNING: not exactly 0.0)"))


(no HF_TOKEN secret attached; public mirrors do not need it)
gate mean|Δ| per case: [0.0, 0.0, 0.0, 0.0, 0.0]
✓ gate passed


In [13]:
# ===== 10. Generate this shard's arms (resumes from an attached earlier run) =====
import os, time, glob, shutil, subprocess, socket, numpy as np
from PIL import Image
VAR = f"{OUT}/variants"; os.makedirs(VAR, exist_ok=True)
GEN_DEADLINE = NB_START + 8.0 * 3600

def count(p): return len(glob.glob(p + "/**/*.png", recursive=True))

def pdiff(tag):
    ds = []
    for f in sorted(glob.glob(f"{VAR}/{tag}/**/*.png", recursive=True))[:8]:
        rel = os.path.relpath(f, f"{VAR}/{tag}")
        a = np.array(Image.open(f).convert("RGB"), float)
        b = np.array(Image.open(f"{GENBASE}/{rel}").convert("RGB").resize(Image.open(f).size), float)
        ds.append(round(float(np.abs(a - b).mean()), 2))
    return ds

for tag, flags in ARMS:
    o = f"{VAR}/{tag}"
    prev = [p for p in glob.glob(f"/kaggle/input/**/gafix_{SHARD}/variants/{tag}", recursive=True) if count(p) >= 200]
    if count(o) < 200 and prev:
        shutil.rmtree(o, ignore_errors=True); shutil.copytree(prev[0], o)
        print(f"{tag}: restored 200 images from an earlier run", flush=True)
    if count(o) >= 200:
        print(f"{tag}: complete ({count(o)} images)", flush=True); continue
    if time.time() > GEN_DEADLINE:
        print(f"GEN_DEADLINE: {tag} not started — save this version and rerun", flush=True); break
    os.makedirs(o, exist_ok=True); t = time.time()
    s = socket.socket(); s.bind(("", 0)); port = s.getsockname()[1]; s.close()
    with open(o + "/log.txt", "w") as lg:
        r = subprocess.run(["/kaggle/temp/freefine_env/bin/torchrun", "--nproc_per_node=2", "--master-port", str(port),
                            "freefine_sweep_2d.py"], cwd=P, env=gen_env(o, f"{GEO}/gen_subset.json", **flags),
                           stdout=lg, stderr=subprocess.STDOUT)
    print(f"[{int((time.time() - NB_START) / 60)}m] {tag}: rc={r.returncode} {int(time.time() - t)}s "
          f"images={count(o)} mean|Δ| vs baseline (first 8) = {pdiff(tag)}", flush=True)
    if r.returncode != 0:
        print("  log tail:", open(o + "/log.txt").read()[-1500:])
done = [t for t, _ in ARMS if count(f"{VAR}/{t}") >= 200]
print("complete arms:", done)
for t in done:
    d = f"{GEO}/gen_eval/{t}"
    if os.path.islink(d): os.remove(d)
    os.symlink(f"{VAR}/{t}", d)


[85m] c0: rc=0 4391s images=200 mean|Δ| vs baseline (first 8) = [0.31, 0.38, 0.23, 0.9, 0.53, 0.1, 0.21, 0.71]
[160m] c0_g2_l4: rc=0 4496s images=200 mean|Δ| vs baseline (first 8) = [1.16, 0.74, 1.54, 1.59, 2.5, 0.38, 0.44, 3.46]
[236m] c0_g2_l8: rc=0 4496s images=200 mean|Δ| vs baseline (first 8) = [1.3, 0.8, 1.74, 1.72, 2.79, 0.41, 0.48, 3.62]
complete arms: ['c0', 'c0_g2_l4', 'c0_g2_l8']


In [14]:
# ===== 6. Download CLIP and DINO once, serially, before the two GPU workers start =====
import subprocess, os
PY = "/kaggle/temp/metric_env/bin/python"
ENV_BASE = os.environ.copy()
ENV_BASE.update({"HF_HOME": "/kaggle/temp/hf", "TORCH_HOME": "/kaggle/temp/torch",
                 "TOKENIZERS_PARALLELISM": "false", "MPLBACKEND": "Agg"})
code_str = ("import clip, torch\n"
            "print('torch', torch.__version__)\n"
            "clip.load('ViT-B/32', device='cpu', jit=False)\n"
            "print('CLIP_PREWARM_OK')\n"
            "torch.hub.load('facebookresearch/dino:main', 'dino_vitb16')\n"
            "print('DINO_PREWARM_OK')\n")
r = subprocess.run([PY, "-c", code_str], env=ENV_BASE, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise RuntimeError("model prewarm failed; do not start the metric workers")
print("✓ CLIP and DINO cached; safe to start GPU workers")


torch 2.6.0+cu124
CLIP_PREWARM_OK
DINO_PREWARM_OK

✓ CLIP and DINO cached; safe to start GPU workers


In [15]:
# ===== 12. Official metrics (seeded), two GPU workers, resumable =====
import os, json, re, glob, subprocess, threading, queue, time
MET = "/kaggle/temp/FreeFine/evaluation/metrics"
EVAL_DEADLINE = NB_START + 11.3 * 3600
RES_PATH = f"{WORK}/gafix_results_{SHARD}.json"
picked = json.load(open(f"{GEO}/subset_meta.json"))
def mem(pred): return [(r["da_n"], r["ins_id"], r["case_id"]) for r in picked if pred(r)]
# task order: FID, IRS, HPS, BGC, SUBC, WRAP_E, MD, FID_DINO, FID_KD
GROUPS = {"rotate_hard": (mem(lambda r: r["edit_type"] == "rotate" and r["difficulty"] == "hard"), "000111100"),
          "resize_hard": (mem(lambda r: r["edit_type"] == "resize" and r["difficulty"] == "hard"), "000111100"),
          "move_all":    (mem(lambda r: r["edit_type"] == "move"), "000111000"),
          "all_200":     (mem(lambda r: True), "100111011")}
SETS = EVAL_REFS + [t for t, _ in ARMS if os.path.exists(f"{GEO}/gen_eval/{t}")]
KEYS = ["FID", "BGC", "SUBC", "WRAP_E", "MD", "FID_DINO", "FID_KD"]

results = {}
for p in glob.glob(f"/kaggle/input/**/gafix_results_{SHARD}.json", recursive=True) + [RES_PATH]:
    if os.path.exists(p):
        for s_, gg in json.load(open(p)).items():
            for g_, v in gg.items():
                if "SUBC" in v: results.setdefault(s_, {})[g_] = v
lock = threading.Lock()

def manifest(setn, gname):
    o = {}
    for d, i, e in GROUPS[gname][0]:
        lf = dict(ann[d]["instances"][i][e]); lf["gen_img_path"] = f"gen_eval/{setn}/{d}/{i}/{e}.png"
        o.setdefault(d, {"instances": {}})["instances"].setdefault(i, {})[e] = lf
    p = f"{GEO}/m_{setn}_{gname}.json"; json.dump(o, open(p, "w")); return p

def run(gpu, setn, gname):
    env = ENV_BASE.copy()
    env.update({"CUDA_VISIBLE_DEVICES": str(gpu), "FF_METRIC_SEED": "42", "FF_MD_SEED": "42",
                "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
    p = subprocess.run([PY, "main.py", "--path", manifest(setn, gname), "--use_relative_path", "--base_dir", GEO,
                        "--fid_path", f"{GEO}/Geo-Bench-2D/source_img_full_v2", "--task", GROUPS[gname][1],
                        "--level", "0"], cwd=MET, env=env, capture_output=True, text=True)
    txt = p.stdout + p.stderr
    open(f"{OUT}/metric_{setn}_{gname}.log", "w").write(txt)
    v = {"n": len(GROUPS[gname][0])}
    for k in KEYS:
        h = re.findall(rf"(?:^|\s){k}:\s*([-\d.eE]+)", txt)
        if h: v[k] = round(float(h[-1]), 4)
    if p.returncode != 0: v["_rc"] = p.returncode; print(txt[-2000:])
    return v

q = queue.Queue()
for s_ in SETS:
    for g_ in GROUPS:
        if g_ not in results.get(s_, {}): q.put((s_, g_))
print("metric jobs to run:", q.qsize(), "| sets:", SETS)

def worker(gpu):
    while time.time() < EVAL_DEADLINE:
        try: s_, g_ = q.get_nowait()
        except queue.Empty: return
        v = run(gpu, s_, g_)
        with lock:
            results.setdefault(s_, {})[g_] = v
            json.dump(results, open(RES_PATH, "w"), indent=2)
        print(f"[GPU{gpu}] {s_:10s} {g_:12s} -> {v}", flush=True)

th = [threading.Thread(target=worker, args=(g,), daemon=True) for g in (0, 1)]
[t.start() for t in th]; [t.join() for t in th]
print("✓ metrics done" if q.empty() else "EVAL_DEADLINE: save this version and rerun to finish")


metric jobs to run: 20 | sets: ['baseline', 'coarse', 'c0', 'c0_g2_l4', 'c0_g2_l8']
[GPU0] baseline   rotate_hard  -> {'n': 22, 'BGC': 0.9664, 'SUBC': 0.8523, 'WRAP_E': 0.0416, 'MD': 14.0096}
[GPU1] baseline   resize_hard  -> {'n': 23, 'BGC': 0.9631, 'SUBC': 0.8391, 'WRAP_E': 0.0585, 'MD': 19.4713}
[GPU0] baseline   move_all     -> {'n': 67, 'BGC': 0.9639, 'SUBC': 0.959, 'WRAP_E': 0.0495}
[GPU0] coarse     rotate_hard  -> {'n': 22, 'BGC': 0.986, 'SUBC': 0.8608, 'WRAP_E': 0.0, 'MD': 17.8582}
[GPU1] baseline   all_200      -> {'n': 200, 'FID': 132.279, 'BGC': 0.9657, 'SUBC': 0.9154, 'WRAP_E': 0.0488, 'FID_DINO': 1636.8248, 'FID_KD': 0.1268}
[GPU1] coarse     move_all     -> {'n': 67, 'BGC': 0.984, 'SUBC': 0.9712, 'WRAP_E': 0.0}
[GPU0] coarse     resize_hard  -> {'n': 23, 'BGC': 0.9808, 'SUBC': 0.8408, 'WRAP_E': 0.0, 'MD': 16.4365}
[GPU1] coarse     all_200      -> {'n': 200, 'FID': 132.6678, 'BGC': 0.9854, 'SUBC': 0.9246, 'WRAP_E': 0.0, 'FID_DINO': 1654.7114, 'FID_KD': 0.1361}
[GPU0] c0 

In [16]:
# ===== 13. Summary =====
import json, pandas as pd
R = json.load(open(RES_PATH))
rows = [{"set": s_, "group": g_, **v} for s_, gg in R.items() for g_, v in gg.items()]
df = pd.DataFrame(rows); df.to_csv(RES_PATH.replace(".json", ".csv"), index=False)
# Values of the same images on the same 200 edits in the earlier evaluations (June studies and the
# exhaustive search agree on all of them). Kernel distance is excluded: it was unseeded then.
EARLIER = {
    ("baseline", "rotate_hard"): {"SUBC": 0.8523, "BGC": 0.9664, "WRAP_E": 0.0416, "MD": 14.0096},
    ("baseline", "resize_hard"): {"SUBC": 0.8391, "BGC": 0.9631, "WRAP_E": 0.0585, "MD": 19.4713},
    ("baseline", "move_all"):    {"SUBC": 0.9590, "BGC": 0.9639, "WRAP_E": 0.0495},
    ("baseline", "all_200"):     {"SUBC": 0.9154, "BGC": 0.9657, "WRAP_E": 0.0488, "FID": 132.279, "FID_DINO": 1636.8248},
    ("coarse", "rotate_hard"):   {"SUBC": 0.8608, "BGC": 0.9860, "WRAP_E": 0.0, "MD": 17.8582},
    ("coarse", "resize_hard"):   {"SUBC": 0.8408, "BGC": 0.9808, "WRAP_E": 0.0, "MD": 16.4365},
    ("coarse", "move_all"):      {"SUBC": 0.9712, "BGC": 0.9840, "WRAP_E": 0.0},
    ("coarse", "all_200"):       {"SUBC": 0.9246, "BGC": 0.9854, "WRAP_E": 0.0, "FID": 132.6678, "FID_DINO": 1654.7114},
}
TOL = {"SUBC": 1.1e-4, "BGC": 1.1e-4, "WRAP_E": 1.1e-4, "MD": 1e-3, "FID": 1e-2, "FID_DINO": 1e-2}
bad = []
for (s_, g_), exp in EARLIER.items():
    if s_ in R and g_ in R[s_]:
        for k, e in exp.items():
            got = R[s_][g_].get(k)
            if got is None or abs(got - e) > TOL[k]:
                bad.append((s_, g_, k, got, e))
print("consistency with the earlier evaluations:", "✓ all values reproduced" if not bad else f"✗ {bad}")
pd.set_option("display.width", 220)
for g_ in ["rotate_hard", "resize_hard", "move_all", "all_200"]:
    print(f"\n=== {g_} ===")
    print(df[df.group == g_].drop(columns=["group"]).set_index("set").to_string())
print(f"\nsaved {RES_PATH} and the .csv; total runtime {(time.time() - NB_START) / 60:.0f} min")


consistency with the earlier evaluations: ✓ all values reproduced

=== rotate_hard ===
           n     BGC    SUBC  WRAP_E       MD  FID  FID_DINO  FID_KD
set                                                                 
baseline  22  0.9664  0.8523  0.0416  14.0096  NaN       NaN     NaN
coarse    22  0.9860  0.8608  0.0000  17.8582  NaN       NaN     NaN
c0        22  0.9667  0.8525  0.0416  14.7950  NaN       NaN     NaN
c0_g2_l4  22  0.9668  0.8370  0.0467  14.7954  NaN       NaN     NaN
c0_g2_l8  22  0.9666  0.8336  0.0474  15.2270  NaN       NaN     NaN

=== resize_hard ===
           n     BGC    SUBC  WRAP_E       MD  FID  FID_DINO  FID_KD
set                                                                 
baseline  23  0.9631  0.8391  0.0585  19.4713  NaN       NaN     NaN
coarse    23  0.9808  0.8408  0.0000  16.4365  NaN       NaN     NaN
c0        23  0.9627  0.8391  0.0584  18.1241  NaN       NaN     NaN
c0_g2_l4  23  0.9630  0.8297  0.0607  17.4692  NaN       NaN    